# Day 16: Qdrant CRUD Operations

## Core Theory (Just-in-Time)

In AI Engineering, a Vector Database like Qdrant acts as the foundational memory layer for Retrieval-Augmented Generation (RAG). While creating collections and querying vectors are common tasks, understanding how to manage the lifecycle of those vectors (Create, Read, Update, Delete - CRUD) with precise metadata is critical for building robust production systems.

### The "Why"
- **Dynamic Data:** Real-world knowledge bases are not static. Documents are updated, user preferences change, and outdated information must be removed to prevent hallucinations.
- **Metadata Filtering:** Vectors alone just provide semantic similarity. Combining them with structured metadata (payloads in Qdrant) enables complex hybrid queries (e.g., 'find similar documents WHERE author=X AND status=active').
- **Consistency:** Ensuring that updates to source documents correctly reflect in the vector store is a major challenge in production AI.

### AI Security Implications
- **PII Protection:** Never blindly upsert raw documents. Always sanitize or redact Personally Identifiable Information (PII) before embedding and storing in Qdrant. Payloads should not contain unmasked user secrets.
- **Prompt Injection:** If document metadata or content is user-generated, treat it as untrusted input. Validate and sanitize fields to prevent injected instructions from poisoning your RAG retrieval context.
- **Fallback Mechanisms:** Vector stores can experience downtime. Implement try/except blocks and provide graceful fallback responses (e.g., retrieving from a cache or returning a standard error message) instead of exposing database stack traces.

### The "How"
We use the `qdrant-client` library in Python to interact with a Qdrant instance. Operations like `upsert` (Create/Update) and `delete` target specific points (vectors) using unique IDs. Payloads are JSON-like objects attached to these points. We will use `pydantic` to enforce strict schemas for our data before it hits the vector database.


## Code Implementation

The following tiered examples demonstrate how to implement Qdrant CRUD operations, progressing from a simple procedural script to a production-grade OOP design with AI Security principles applied.


In [1]:
# Basic Implementation: Procedural CRUD with minimal boilerplate
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

# 1. Initialize in-memory client and collection
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="basic_docs",
    vectors_config=VectorParams(size=4, distance=Distance.COSINE)
)

# 2. Create (Insert)
point_id = str(uuid.uuid4())
client.upsert(
    collection_name="basic_docs",
    points=[
        PointStruct(id=point_id, vector=[0.1, 0.2, 0.3, 0.4], payload={"title": "Basic Doc", "author": "Alice"})
    ]
)
print(f"Basic: Inserted {point_id}")

# 3. Read
points = client.retrieve(collection_name="basic_docs", ids=[point_id], with_payload=True)
print(f"Basic: Retrieved payload - {points[0].payload}")

# 4. Update Payload
client.set_payload(collection_name="basic_docs", payload={"status": "archived"}, points=[point_id])
print("Basic: Updated payload")

# 5. Delete
client.delete(collection_name="basic_docs", points_selector=[point_id])
print("Basic: Deleted point")


Basic: Inserted 717a0ef3-a559-4440-9740-f4dd213c5a82
Basic: Retrieved payload - {'title': 'Basic Doc', 'author': 'Alice'}
Basic: Updated payload
Basic: Deleted point


In [2]:
# Medium Implementation: Clean OOP and state management
import uuid
from typing import List, Dict, Any, Optional
from pydantic import BaseModel
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

class MediumDocumentMetadata(BaseModel):
    """Schema for document metadata."""
    title: str
    author: str
    status: str = "active"

class MediumQdrantManager:
    """Manages CRUD operations for a specific Qdrant collection."""
    def __init__(self, collection_name: str):
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name
        
        if not self.client.collection_exists(collection_name=self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=4, distance=Distance.COSINE)
            )

    def insert(self, doc_id: str, vector: List[float], metadata: MediumDocumentMetadata) -> None:
        point = PointStruct(id=doc_id, vector=vector, payload=metadata.model_dump())
        self.client.upsert(collection_name=self.collection_name, points=[point])
        print(f"Medium: Inserted {doc_id}")

    def read(self, doc_id: str) -> Optional[Dict[str, Any]]:
        points = self.client.retrieve(collection_name=self.collection_name, ids=[doc_id], with_payload=True)
        return points[0].payload if points else None

# Execution
if __name__ == "__main__":
    manager = MediumQdrantManager("medium_docs")
    doc_id = str(uuid.uuid4())
    
    manager.insert(doc_id, [0.5, 0.5, 0.5, 0.5], MediumDocumentMetadata(title="OOP Guide", author="Bob"))
    print(f"Medium: Read result: {manager.read(doc_id)}")


Medium: Inserted da7b9f57-d16e-40be-b8e4-ebd82fa32642
Medium: Read result: {'title': 'OOP Guide', 'author': 'Bob', 'status': 'active'}


In [3]:
# Advanced Implementation: Production-ready with robust error handling, PII masking, and explicit typing
import uuid
import logging
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field, ValidationError
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

class DocumentMetadata(BaseModel):
    """Strict schema with simulated PII protection."""
    title: str = Field(..., description="Document title")
    author: str = Field(..., description="Document author")
    email: Optional[str] = Field(default=None, description="Author email (will be masked)")
    status: str = Field(default="active", description="Status")

class QdrantManager:
    """Production-grade CRUD manager with fallback mechanisms."""
    def __init__(self, collection_name: str, vector_size: int = 4):
        try:
            self.client = QdrantClient(":memory:")
            self.collection_name = collection_name
            self.vector_size = vector_size
            
            if not self.client.collection_exists(collection_name=self.collection_name):
                self.client.create_collection(
                    collection_name=self.collection_name,
                    vectors_config=VectorParams(size=self.vector_size, distance=Distance.COSINE)
                )
                logger.info(f"Collection '{self.collection_name}' created.")
        except Exception as e:
            logger.error(f"Failed to initialize Qdrant client: {e}")
            raise

    def _mask_pii(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """Redacts sensitive information before storage."""
        safe_payload = payload.copy()
        if "email" in safe_payload and safe_payload["email"]:
            safe_payload["email"] = "[REDACTED]"
        return safe_payload

    def insert_point(self, point_id: str, vector: List[float], payload: DocumentMetadata) -> bool:
        """Inserts a point with try/except fallback and PII masking."""
        try:
            safe_payload = self._mask_pii(payload.model_dump())
            point = PointStruct(id=point_id, vector=vector, payload=safe_payload)
            self.client.upsert(collection_name=self.collection_name, points=[point])
            logger.info(f"Inserted point {point_id} successfully.")
            return True
        except Exception as e:
            logger.error(f"Fallback: Failed to insert point {point_id} due to {e}")
            return False

    def read_point(self, point_id: str) -> Optional[Dict[str, Any]]:
        """Retrieves a point safely."""
        try:
            points = self.client.retrieve(collection_name=self.collection_name, ids=[point_id], with_payload=True)
            if not points:
                logger.warning(f"Point {point_id} not found.")
                return None
            return points[0].payload
        except Exception as e:
            logger.error(f"Fallback: Failed to read point {point_id}: {e}")
            return None

    def update_payload(self, point_id: str, new_payload: dict) -> None:
        """Updates specific fields in the metadata payload."""
        try:
            operation_info = self.client.set_payload(
                collection_name=self.collection_name,
                payload=new_payload,
                points=[point_id]
            )
            logger.info(f"Updated payload for {point_id}. Status: {operation_info.status.name}")
        except Exception as e:
            logger.error(f"Fallback: Failed to update payload for {point_id}: {e}")

    def delete_point(self, point_id: str) -> None:
        """Removes a point safely."""
        try:
            operation_info = self.client.delete(
                collection_name=self.collection_name,
                points_selector=[point_id]
            )
            logger.info(f"Deleted point {point_id}. Status: {operation_info.status.name}")
        except Exception as e:
            logger.error(f"Fallback: Failed to delete point {point_id}: {e}")

# Execution
if __name__ == "__main__":
    adv_manager = QdrantManager(collection_name="advanced_docs")
    doc_id = str(uuid.uuid4())
    
    # Simulate data with PII
    try:
        secure_meta = DocumentMetadata(title="Security Policies", author="Charlie", email="charlie@example.com")
        
        # 1. Insert
        adv_manager.insert_point(doc_id, [0.1, 0.9, 0.2, 0.8], secure_meta)
        
        # 2. Read and verify masking
        retrieved = adv_manager.read_point(doc_id)
        logger.info(f"Retrieved Payload (Note the redacted email): {retrieved}")
        
    except ValidationError as ve:
        logger.error(f"Data validation error: {ve}")


2026-08-21 15:03:18,867 - INFO - Collection 'advanced_docs' created.


2026-08-21 15:03:18,868 - INFO - Inserted point 10e00922-e939-4a5c-b1e2-88a35c9eed54 successfully.


2026-08-21 15:03:18,869 - INFO - Retrieved Payload (Note the redacted email): {'title': 'Security Policies', 'author': 'Charlie', 'email': '[REDACTED]', 'status': 'active'}


## Common Pitfalls in Production

1. **Dangling Vectors (Zombie Data):** Deleting a document from your primary database (e.g., Postgres) but forgetting to issue a delete command to Qdrant. This leads to the LLM answering questions based on deleted or revoked information.
2. **Schema Drift:** Modifying the structure of the metadata payloads in your application without ensuring historical payloads in the vector DB are compatible, causing crashes during filtering or retrieval.
3. **Costly Complete Updates:** Re-embedding the entire document text just to update a small metadata field (like `status=active` to `status=archived`). You should use Qdrant's payload update mechanisms (`set_payload`) instead of full `upsert` when the text hasn't changed.

4. **PII Leakage in Metadata:** Storing unmasked PII (like emails or SSNs) in the vector database payload. Always filter or redact PII at the application layer before embedding and storing data.


## Practical Lab / Homework

**Task:** Expand upon the CRUD operations by implementing a batch upsert mechanism.
1. Create a method `batch_insert_points` in a new class `BatchQdrantManager`.
2. It should accept a list of IDs, a list of vectors, and a list of `DocumentMetadata` objects.
3. Execute a single batch upsert call to Qdrant.
4. Verify the operation by retrieving all inserted points.
5. Record a brief async video walkthrough of your design decisions, specifically explaining how batching improves database efficiency and how your schema avoids schema drift.

**Constraint:** Provide a fully working script with strict type hinting. Do not use pseudo-code.


In [4]:
from qdrant_client.models import Batch

class BatchQdrantManager(QdrantManager):
    """Extends the QdrantManager to support batch operations."""
    
    def batch_insert_points(self, point_ids: List[str], vectors: List[List[float]], payloads: List[DocumentMetadata]) -> None:
        """
        Performs a batch upsert of multiple points simultaneously.
        """
        if not (len(point_ids) == len(vectors) == len(payloads)):
            raise ValueError("Mismatched lengths of IDs, vectors, and payloads.")
            
        # Convert Pydantic models to dicts
        payload_dicts = [p.model_dump() for p in payloads]
        
        operation_info = self.client.upsert(
            collection_name=self.collection_name,
            points=Batch(
                ids=point_ids,
                vectors=vectors,
                payloads=payload_dicts
            )
        )
        print(f"Batch upserted {len(point_ids)} points. Status: {operation_info.status.name}")

    def read_multiple_points(self, point_ids: List[str]) -> List[Dict[str, Any]]:
        """
        Retrieves multiple points by their IDs.
        """
        points = self.client.retrieve(
            collection_name=self.collection_name,
            ids=point_ids,
            with_payload=True
        )
        
        results = []
        for p in points:
            print(f"Retrieved point {p.id}: Payload = {p.payload}")
            if p.payload:
                results.append(p.payload)
        return results

# Lab Solution Execution
if __name__ == "__main__":
    batch_manager = BatchQdrantManager(collection_name="batch_engineering_docs")
    
    # Generate dummy data
    ids = [str(uuid.uuid4()) for _ in range(3)]
    vecs = [
        [0.1, 0.1, 0.1, 0.1],
        [0.2, 0.2, 0.2, 0.2],
        [0.3, 0.3, 0.3, 0.3]
    ]
    docs = [
        DocumentMetadata(title="Doc A", author="Alice"),
        DocumentMetadata(title="Doc B", author="Bob"),
        DocumentMetadata(title="Doc C", author="Charlie")
    ]
    
    print("\n--- Batch Create ---")
    batch_manager.batch_insert_points(point_ids=ids, vectors=vecs, payloads=docs)
    
    print("\n--- Verify Batch Insertion ---")
    batch_manager.read_multiple_points(point_ids=ids)


2026-08-21 15:03:18,900 - INFO - Collection 'batch_engineering_docs' created.



--- Batch Create ---
Batch upserted 3 points. Status: COMPLETED

--- Verify Batch Insertion ---
Retrieved point cb2184c6-bc89-4a44-9a9a-8cc2d83911a3: Payload = {'title': 'Doc A', 'author': 'Alice', 'email': None, 'status': 'active'}
Retrieved point 593f1931-5723-4ead-b7ce-437c3e3be72c: Payload = {'title': 'Doc B', 'author': 'Bob', 'email': None, 'status': 'active'}
Retrieved point 90a617eb-b5a2-4065-ab46-b0dea03d45ac: Payload = {'title': 'Doc C', 'author': 'Charlie', 'email': None, 'status': 'active'}


## Reference Links

- [Qdrant Python Client Documentation](https://qdrant.tech/documentation/interfaces/python/)
- [Qdrant Concepts: Points and Vectors](https://qdrant.tech/documentation/concepts/points/)
- [Pydantic Official Documentation](https://docs.pydantic.dev/latest/)
